# EPP Detector — Inferencia con YOLO26

Detección de Equipos de Protección Personal (EPP) en obras de construcción.

**Requisitos previos:**
- Activar GPU: `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`

---

## VERIFICACIONES IMPORTACIONES Y CARGA DE MODELO

In [ ]:
!nvidia-smi

%pip install -q "ultralytics>=8.4.0" supervision

# Desactivar telemetría de Ultralytics
!yolo settings sync=False

import ultralytics
ultralytics.checks()

#CARGA DEL MODELO
import os
from ultralytics import YOLO

HOME = os.getcwd()

# ── URL directa del Release en GitHub ──────────────────────
MODEL_URL = "https://github.com/HoracioMantilla/MAIC1125-HMJ/releases/download/v1.0/best.pt"
MODEL_PATH = f"{HOME}/best.pt"
# ────────────────────────────────────────────────────────────

# Descargar solo si no existe ya en el entorno
if not os.path.exists(MODEL_PATH):
    print("Descargando modelo desde GitHub Releases...")
    !wget -q --show-progress -O {MODEL_PATH} {MODEL_URL}
    print("Descarga completada.")
else:
    print("Modelo ya presente en el entorno, omitiendo descarga.")

model = YOLO(MODEL_PATH)
print(f"\nModelo cargado correctamente.")
print(f"Clases: {model.names}")

# Definir función de anotación

import supervision as sv
from PIL import Image


def annotate(image: Image.Image, detections: sv.Detections) -> Image.Image:
    color = sv.ColorPalette.from_hex([
        "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00",
        "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    ])

    text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)

    box_annotator = sv.BoxAnnotator(color=color)
    label_annotator = sv.LabelAnnotator(
        color=color,
        text_color=sv.Color.BLACK,
        text_scale=text_scale,
        smart_position=True
    )

    out = image.copy()
    out = box_annotator.annotate(out, detections)
    out = label_annotator.annotate(out, detections)
    out.thumbnail((1000, 1000))
    return out


print('Función annotate() lista.')



## Inferencia sobre imagen aleatoria del repositorio

Descarga una imagen al azar desde la carpeta `5_images` del repo y ejecuta la detección.

In [ ]:
import requests
import random
from PIL import Image
from io import BytesIO

# ── Configuración del repo ──────────────────────────────────
GITHUB_USER = "HoracioMantilla"
GITHUB_REPO = "MAIC1125-HMJ"
GITHUB_BRANCH = "main"
IMAGES_FOLDER = "5_images"
# ────────────────────────────────────────────────────────────

# Obtener listado de archivos de la carpeta via GitHub API
api_url = f"https://api.github.com/repos/{GITHUB_USER}/{GITHUB_REPO}/contents/{IMAGES_FOLDER}?ref={GITHUB_BRANCH}"
response = requests.get(api_url)
response.raise_for_status()

# Filtrar solo imágenes
EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
image_files = [
    f for f in response.json()
    if f['name'].lower().endswith(EXTENSIONS)
]

if not image_files:
    raise FileNotFoundError(f"No se encontraron imágenes en '{IMAGES_FOLDER}'. Verificá el nombre de la carpeta.")

# Elegir una al azar
chosen = random.choice(image_files)
print(f"Imagen seleccionada aleatoriamente: {chosen['name']}")
print(f"URL: {chosen['download_url']}")

# Descargar la imagen
img_response = requests.get(chosen['download_url'])
img_response.raise_for_status()
image = Image.open(BytesIO(img_response.content)).convert("RGB")

# Inferencia
result = model.predict(image, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)
annotated_image = annotate(image, detections)

print(f"Detecciones encontradas: {len(detections)}")
annotated_image

## Inferencia sobre una imagen propia

Suba una imagen desde su computadora cuando se le solicite.

In [ ]:
from google.colab import files

print('Selecciona una imagen para analizar...')
uploaded = files.upload()

image_name = list(uploaded.keys())[0]
image_path = f'{HOME}/{image_name}'

with open(image_path, 'wb') as f:
    f.write(uploaded[image_name])

# Inferencia
image = Image.open(image_path)
result = model.predict(image, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)
annotated_image = annotate(image, detections)

print(f'Detecciones encontradas: {len(detections)}')
annotated_image